In [28]:
import dynaconf

settings = dynaconf.Dynaconf(settings_files=["settings.toml"], environments=True)
settings["absorption_database_error_handling"]


<Box: {'x': {'missing': 'ignore', 'scalar': 'ignore', 'bounds': 'raise'}, 'p': {'missing': 'raise', 'scalar': 'raise', 'bounds': 'ignore'}, 't': {'missing': 'raise', 'scalar': 'raise', 'bounds': 'ignore'}}>

In [37]:
import enum
import warnings
from collections.abc import Mapping

import attrs

from eradiate.exceptions import InterpolationError


class ErrorHandlingAction(enum.Enum):
    IGNORE = "ignore"
    RAISE = "raise"
    WARN = "warn"


@attrs.define
class ErrorHandlingPolicy:
    missing: ErrorHandlingAction
    scalar: ErrorHandlingAction
    bounds: ErrorHandlingAction

    @classmethod
    def convert(cls, value):
        if isinstance(value, Mapping):
            kwargs = {k: ErrorHandlingAction(v) for k, v in value.items()}
            return cls(**kwargs)
        else:
            return value


@attrs.define
class ErrorHandlingConfiguration:
    x: ErrorHandlingPolicy = attrs.field(converter=ErrorHandlingPolicy.convert)
    p: ErrorHandlingPolicy = attrs.field(converter=ErrorHandlingPolicy.convert)
    t: ErrorHandlingPolicy = attrs.field(converter=ErrorHandlingPolicy.convert)

    @classmethod
    def convert(cls, value):
        if isinstance(value, dict):
            return cls(**value)
        else:
            return value


def handle_error(error: InterpolationError, action: ErrorHandlingAction):
    if action is ErrorHandlingAction.IGNORE:
        return

    if action is ErrorHandlingAction.WARN:
        warnings.warn(str(error), UserWarning)
        return

    if action is ErrorHandlingAction.RAISE:
        raise error

    raise NotImplementedError


config = ErrorHandlingConfiguration.convert(
    settings["absorption_database_error_handling"]
)
config


ErrorHandlingConfiguration(x=ErrorHandlingPolicy(missing=<ErrorHandlingAction.IGNORE: 'ignore'>, scalar=<ErrorHandlingAction.IGNORE: 'ignore'>, bounds=<ErrorHandlingAction.RAISE: 'raise'>), p=ErrorHandlingPolicy(missing=<ErrorHandlingAction.RAISE: 'raise'>, scalar=<ErrorHandlingAction.RAISE: 'raise'>, bounds=<ErrorHandlingAction.IGNORE: 'ignore'>), t=ErrorHandlingPolicy(missing=<ErrorHandlingAction.RAISE: 'raise'>, scalar=<ErrorHandlingAction.RAISE: 'raise'>, bounds=<ErrorHandlingAction.IGNORE: 'ignore'>))